In [ ]:
import csv
import pandas as pd
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore') # Does not work for this kind of warning
pd.set_option('display.max_rows', None)


In [ ]:
# Introducing part time Cezanne
def generate_dynamic_deficit_schedule_with_midwife(start_date_str, total_weeks=26):
    # Full-time team members
    ft_workers = [
        "Dr. Alice Smith", "Nurse Bob Jones", "Dr. Charlie Brown", "Nurse Diana Prince",
        "Dr. Evan Wright", "Nurse Fiona Gallagher", "Dr. George Clark", "Nurse Hannah Abbott"
    ]
    # Part-time team member
    pt_workers = ["Midwife Cezanne"]
    all_workers = ft_workers + pt_workers
    
    PUBLIC_HOLIDAYS = {
        "2026-10-12", "2026-11-11", "2026-11-26", "2026-12-25", "2027-01-01", "2027-01-19"
    }

    staff_vacations = {w: set() for w in all_workers}
    staff_vacations["Dr. Alice Smith"] = {"2026-09-24", "2026-09-25"}
    staff_vacations["Dr. Charlie Brown"] = {"2026-10-01", "2026-10-02"}
    staff_vacations["Dr. Evan Wright"] = {"2026-12-24", "2026-12-26"}
    
    global_counts = {w: {'Day_Shifts': 0, 'Night_Shifts': 0, 'Full_Clinic': 0, 'Half_Clinic': 0, 'Total_Hours': 0} for w in all_workers}
    deficit_hours = {w: 0.0 for w in all_workers}
    
    schedule_data = []
    last_night_worker = None
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")

    for week_idx in range(total_weeks):
        week_start_date = start_date + timedelta(weeks=week_idx)
        weekly_hospital_counts = {w: 0 for w in all_workers}
        week_days_manifest = {}
        
        has_holiday_this_week = False
        for day_offset in range(7):
            if (week_start_date + timedelta(days=day_offset)).strftime("%Y-%m-%d") in PUBLIC_HOLIDAYS:
                has_holiday_this_week = True

        # --- PHASE 1: Assign 12-Hour Hospital Shifts ---
        for day_offset in range(7):
            current_date = week_start_date + timedelta(days=day_offset)
            date_str = current_date.strftime("%Y-%m-%d")
            day_name = current_date.strftime("%A")
            
            week_days_manifest[date_str] = {
                'day_name': day_name, 'day_hospital': None, 'night_hospital': None, 
                'full_clinic_staff': [], 'half_clinic_staff': []
            }
            
            active_today = [w for w in all_workers if date_str not in staff_vacations[w]]
            
            # --- Day Shift ---
            available_for_day = [
                w for w in active_today 
                if w != last_night_worker and (
                    (w in ft_workers and weekly_hospital_counts[w] < 3) or 
                    (w in pt_workers and weekly_hospital_counts[w] < 1) # Part-time cap
                )
            ]
            available_for_day.sort(key=lambda w: (weekly_hospital_counts[w], global_counts[w]['Day_Shifts'], global_counts[w]['Total_Hours']))
            assigned_day_worker = available_for_day[0] if available_for_day else None
            
            if assigned_day_worker:
                week_days_manifest[date_str]['day_hospital'] = assigned_day_worker
                weekly_hospital_counts[assigned_day_worker] += 1
                global_counts[assigned_day_worker]['Day_Shifts'] += 1
                global_counts[assigned_day_worker]['Total_Hours'] += 12
            
            # --- Night Shift ---
            available_for_night = [
                w for w in active_today 
                if w != assigned_day_worker and w != last_night_worker and (
                    (w in ft_workers and weekly_hospital_counts[w] < 3) or 
                    (w in pt_workers and weekly_hospital_counts[w] < 1) # Part-time cap
                )
            ]
            available_for_night.sort(key=lambda w: (weekly_hospital_counts[w], global_counts[w]['Night_Shifts'], global_counts[w]['Total_Hours']))
            assigned_night_worker = available_for_night[0] if available_for_night else None
            
            if assigned_night_worker:
                week_days_manifest[date_str]['night_hospital'] = assigned_night_worker
                weekly_hospital_counts[assigned_night_worker] += 1
                global_counts[assigned_night_worker]['Night_Shifts'] += 1
                global_counts[assigned_night_worker]['Total_Hours'] += 12
            
            last_night_worker = assigned_night_worker

        # --- PHASE 2: Dynamic Clinic Assignment & Deficit Logic ---
        for worker in all_workers:
            hospital_hours = weekly_hospital_counts[worker] * 12
            
            # Dynamic targets based on status (Full-Time vs. Midwife Cezanne Part-Time)
            if worker in pt_workers:
                base_target = 12 if has_holiday_this_week else 20
                safety_max = 32
            else:
                base_target = 32 if has_holiday_this_week else 40
                safety_max = 60
                
            personal_target = base_target + deficit_hours[worker]
            needed_clinic_hours = max(0, personal_target - hospital_hours)
            
            if hospital_hours + needed_clinic_hours > safety_max:
                needed_clinic_hours = safety_max - hospital_hours
                
            req_full = int(needed_clinic_hours // 8)
            req_half = int((needed_clinic_hours % 8) // 4)
            
            full_assigned = 0
            half_assigned = 0
            
            for date_str, day_data in week_days_manifest.items():
                if full_assigned == req_full and half_assigned == req_half:
                    break
                
                if day_data['day_name'] in ['Saturday', 'Sunday'] or date_str in PUBLIC_HOLIDAYS or date_str in staff_vacations[worker]:
                    continue
                
                if day_data['day_hospital'] != worker and day_data['night_hospital'] != worker:
                    prev_date_str = (datetime.strptime(date_str, "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d")
                    was_resting = prev_date_str in week_days_manifest and week_days_manifest[prev_date_str]['night_hospital'] == worker
                    
                    if not was_resting:
                        if full_assigned < req_full:
                            day_data['full_clinic_staff'].append(worker)
                            full_assigned += 1
                            global_counts[worker]['Full_Clinic'] += 1
                            global_counts[worker]['Total_Hours'] += 8
                        elif half_assigned < req_half:
                            day_data['half_clinic_staff'].append(worker)
                            half_assigned += 1
                            global_counts[worker]['Half_Clinic'] += 1
                            global_counts[worker]['Total_Hours'] += 4

            hours_worked_this_week = hospital_hours + (full_assigned * 8) + (half_assigned * 4)
            deficit_hours[worker] = max(0, personal_target - hours_worked_this_week)

        # --- PHASE 3: Compile Manifest ---
        for date_str, day_data in sorted(week_days_manifest.items()):
            is_holiday = date_str in PUBLIC_HOLIDAYS
            schedule_data.append({
                'Date': date_str,
                'Day of Week': day_data['day_name'] + (" 🎉 [HOLIDAY]" if is_holiday else ""),
                'Day Shift (12hr)': day_data['day_hospital'] if day_data['day_hospital'] else "None",
                'Night Shift (12hr)': day_data['night_hospital'] if day_data['night_hospital'] else "None",
                'Full Clinic (8hr)': "🏥 CLOSED" if is_holiday else (", ".join(day_data['full_clinic_staff']) if day_data['full_clinic_staff'] else "None"),
                'Half Clinic (4hr)': "🏥 CLOSED" if is_holiday else (", ".join(day_data['half_clinic_staff']) if day_data['half_clinic_staff'] else "None")
            })

    csv_filename = "hospital_holiday_40hr_schedule.csv"
    fields = ['Date', 'Day of Week', 'Day Shift (12hr)', 'Night Shift (12hr)', 'Full Clinic (8hr)', 'Half Clinic (4hr)']
    with open(csv_filename, mode='w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(schedule_data)
        
    print(f"Schedule exported successfully to {csv_filename}!\n")
    print("6-Month Target Verification Summary:")
    print(f"{'Worker Name':<22} | {'Day Shifts':<10} | {'Night Calls':<11} | {'Full Clinic':<11} | {'Half Clinic':<11} | {'Total Hours':<12}")
    print("-" * 92)
    for worker, counts in sorted(global_counts.items()):
        print(f"{worker:<22} | {counts['Day_Shifts']:<10} | {counts['Night_Shifts']:<11} | {counts['Full_Clinic']:<11} | {counts['Half_Clinic']:<11} | {counts['Total_Hours']:<12} hrs")

generate_dynamic_deficit_schedule_with_midwife("2026-09-20")
